# Bhaashik · NLLB adaptation on Colab / Kaggle

A complete **baseline → finetune → evaluate → error review** workflow for Bhojpuri, Maithili, Magahi, Awadhi, Braj, Nepali and Marwari. Uses a standard HF encoder–decoder, not a chat LLM.

Use the first setup cell with `SETUP_MODE='base'` while preparing data on CPU. For model inference and training, select a GPU runtime (T4 or better), restart the session, and change `SETUP_MODE='train'`; the installer will verify CUDA. Enable Internet in Kaggle and run in order. Free GPU availability and memory are not guaranteed. NLLB weights are CC-BY-NC 4.0. Accept upstream terms when required. Never paste tokens into cells. Example sentences are software fixtures, not research gold.

No notebook has been run on a hosted GPU as part of this repository's offline validation. The local integration tests exercise the same training, export and prediction functions with a tiny random model.

Setup supports Python 3.10–3.13. Python 3.13 uses SentencePiece 0.2.1 wheels; older supported Python versions retain 0.2.0. The shared installer does not install compilers or invoke sudo. Restart the kernel before setup if model libraries have already been imported.


In [ ]:
from pathlib import Path
import os, subprocess, sys
# CPU preparation by default. Change to 'train' when GPU training begins.
SETUP_MODE = 'base'  # base | preflight | train | train,repairs
REQUIRE_CUDA = SETUP_MODE.startswith('train')
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
ROOT = Path('/kaggle/working/BhaashikMTFinetuning' if Path('/kaggle/working').exists() else '/content/BhaashikMTFinetuning')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/bhaashik/BhaashikMTFinetuning.git',str(ROOT)], check=True)
os.chdir(ROOT)
# Use the shared installer: compatible wheels, visible pip errors, no sudo fallback.
installer = ROOT/'scripts/setup_notebook.py'
assert installer.is_file(), 'This checkout is old. Save local work, then refresh/reclone the updated repository.'
loaded = [name for name in ('torch','transformers','sentencepiece','peft','datasets') if name in sys.modules]
if loaded:
    raise RuntimeError('Restart the notebook session/kernel before installation; already imported: '+', '.join(loaded))
command = [sys.executable, str(installer), '--extras', SETUP_MODE]
if REQUIRE_CUDA:
    command.append('--require-cuda')
subprocess.run(command, check=True)
# The active notebook kernel predates pip's new editable-install .pth file.
# Expose this checkout immediately without requiring a second kernel restart.
project_src = str(ROOT / 'src')
if project_src not in sys.path:
    sys.path.insert(0, project_src)
import bhaashik
print('Imported Bhaashik package from:', bhaashik.__file__)
if REQUIRE_CUDA:
    import torch
    print('Notebook GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU data-preparation setup complete; switch to train + GPU for training cells.')


## Choose a direction and data

Choose Bhojpuri (`bho_Deva`), Maithili (`mai_Deva`), Magahi (`mag_Deva`), Awadhi (`awa_Deva`), Nepali (`npi_Deva`), Braj (`bra_Deva`, project-local), or Marwari (`mwr_Deva`, project-local). Hindi (`hin_Deva`) and English (`eng_Latn`) may be bridge languages. Native NLLB does **not** support Braj or Marwari. Check model-specific language inventories before inferring support.

For research set `USE_EXAMPLE_DATA=False`, supply licensed, document-disjoint `train.jsonl`, `dev.jsonl` and `test.jsonl` under `data/local/{PAIR}`, and run the corpus diagnostics before downloading large models. `USE_EXAMPLE_DATA=True` is only a two-example-per-split mechanics test, not a meaningful benchmark.


In [ ]:
import json
SRC, TGT = 'bho_Deva', 'hin_Deva'
SEED, MODE = 42, 'lora'  # Repeat with seeds 13, 42, 87; compare full separately.
MODEL = 'facebook/nllb-200-distilled-600M'
REVISION = None  # Set an immutable Hub commit for final experiments.
USE_EXAMPLE_DATA = True  # Only a mechanics demonstration, not a research result.
PAIR = f"{SRC.split('_')[0]}-{TGT.split('_')[0]}"
if USE_EXAMPLE_DATA and (SRC, TGT) != ('bho_Deva', 'hin_Deva'):
    raise ValueError('Example data are Bhojpuri → Hindi only; supply matching data.')
DATA = Path('data/examples') if USE_EXAMPLE_DATA else Path('data/local') / PAIR
RUN = Path('runs') / f'{PAIR}-{MODE}-s{SEED}'
from bhaashik.data import audit
print(audit({s:DATA / f'{s}.jsonl' for s in ('train','dev','test')}))


## Pre-training corpus diagnostics (all seven focus languages)

Evaluate the dataset **before** finetuning. Diagnostics check train/dev/test ID/text/document leakage, sentence lengths, source/target ratios, script distributions, provenance coverage, near-duplicate candidates and suspicious characters. Near-duplicate detection is heuristic, not a guarantee of semantic independence; inspect warnings and sample translations manually. The test set is checked for leakage **but never used to choose the model or training settings**.


In [ ]:
from bhaashik.pretrain_review import inspect_splits, plot_review
from bhaashik.language_support import language_options, require_native_nllb

REVIEW_DIR = ROOT / "runs" / f"{PAIR}-pretrain-review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("Language coverage (native != paper claim):")
for item in language_options():
    print(item)
CORPUS_REVIEW = inspect_splits(
    {split: DATA / f"{split}.jsonl" for split in ("train", "dev", "test")},
    REVIEW_DIR,
)
print("Data audit ready for review:", CORPUS_REVIEW["ready_for_training"])
for issue in CORPUS_REVIEW["issues"]:
    print(issue["severity"].upper(), issue["check"], issue["message"])
for chart in plot_review(REVIEW_DIR):
    print("Saved figure:", chart)
    try:
        from IPython.display import display, Image
        display(Image(filename=str(chart)))
    except ImportError:
        pass


## Unfinetuned baselines on development data

First freeze a reproducible sample of **development** rows (never test rows). Run the same sample through NLLB and any optional baseline models. Model downloads may take time; set `RUN_NLLB_DEV_BASELINE=False` during CPU-only corpus review. Native NLLB is available for Bhojpuri, Maithili, Magahi, Awadhi and Nepali, but not Braj or Marwari; unsupported directions require an explicitly trained language extension. Maithili/Nepali are supported in IndicTrans2. BhashaVerse 36-language claims exceed the checkpoint's explicit Hub language list, so the adapter permits only card-verified languages.


In [ ]:
from bhaashik.model_comparisons import select_dev_sample
from bhaashik.language_support import require_native_nllb, LANGUAGE_BY_TAG

DEV_SAMPLE_SIZE = 100   # small, fixed development sample for cross-model comparisons
DEV_SAMPLE = REVIEW_DIR / "dev_comparison_sample.jsonl"
if not DEV_SAMPLE.exists():
    select_dev_sample(DATA / "dev.jsonl", DEV_SAMPLE, sample_size=DEV_SAMPLE_SIZE, seed=SEED)
else:
    from bhaashik.data import file_hash
    sample_manifest = json.loads(Path(str(DEV_SAMPLE) + ".manifest.json").read_text())
    assert sample_manifest["source_sha256"] == file_hash(DATA / "dev.jsonl"), (
        "Development data changed; use a new run/review directory and sample"
    )
print("Frozen dev sample:", DEV_SAMPLE)
NLLB_SUPPORTED = all(\n    code not in LANGUAGE_BY_TAG or LANGUAGE_BY_TAG[code].nllb_native\n    for code in (SRC, TGT)\n)\nRUN_NLLB_DEV_BASELINE = SETUP_MODE.startswith("train") and NLLB_SUPPORTED
DEV_PREDICTIONS = {}
if RUN_NLLB_DEV_BASELINE:
    require_native_nllb(SRC, TGT)
    from bhaashik.modeling import translate
    NLLB_DEV = REVIEW_DIR / "nllb-dev.jsonl"
    if not NLLB_DEV.exists():
        translate(MODEL, SRC, TGT, DEV_SAMPLE, NLLB_DEV, revision=REVISION,
                  batch_size=1, max_length=128, max_new_tokens=128, beams=4)
    DEV_PREDICTIONS["NLLB"] = NLLB_DEV
else:
    print("NLLB dev baseline skipped (CPU review mode or unsupported native language tag).")


## Optional IndicTrans2 and BhashaVerse comparators

Both are **off by default** and use the exact frozen dev sample. IndicTrans2 requires `IndicTransToolkit`, supported native tags and a separate upstream gated-model/custom-code installation if applicable; inspect the model code before enabling `ALLOW_INDIC_REMOTE_CODE`. BhashaVerse requires a previously downloaded and reviewed `hf_inference.py` from the model's Hugging Face repository, with its required dictionary, SentencePiece and weights. It must **not** use the standard MBart/AutoTokenizer path. The model adapters write prediction JSONL files with matching IDs for fair scoring. Do not install their incompatible dependency stacks into an already-running NLLB experiment without validating compatibility; outputs can be generated in separate Kaggle/Colab sessions and supplied as files here.


In [ ]:
from bhaashik.model_comparisons import (
    run_indictrans2_dev, run_bhashaverse_dev,
)
from bhaashik.language_support import LANGUAGE_BY_TAG

RUN_INDICTRANS2 = False
ALLOW_INDIC_REMOTE_CODE = False
RUN_BHASHAVERSE = False
BHASHAVERSE_INFERENCE_SCRIPT = None  # Path to reviewed upstream hf_inference.py
# Optional: add existing externally generated predictions here:
EXTERNAL_DEV_PREDICTIONS = {}  # e.g. {"IndicTrans2": REVIEW_DIR / "indictrans2-dev.jsonl"}

if RUN_INDICTRANS2:
    output = REVIEW_DIR / "indictrans2-dev.jsonl"
    if not output.exists():
        run_indictrans2_dev(
            DEV_SAMPLE, output, SRC, TGT,
            allow_remote_code=ALLOW_INDIC_REMOTE_CODE, batch_size=4,
        )
    DEV_PREDICTIONS["IndicTrans2"] = output

if RUN_BHASHAVERSE:
    if BHASHAVERSE_INFERENCE_SCRIPT is None:
        raise ValueError("Provide a reviewed upstream hf_inference.py file path")
    output = REVIEW_DIR / "bhashaverse-dev.jsonl"
    if not output.exists():
        run_bhashaverse_dev(
            DEV_SAMPLE, output, SRC, TGT,
            inference_script=BHASHAVERSE_INFERENCE_SCRIPT, batch_size=4,
        )
    DEV_PREDICTIONS["BhashaVerse"] = output

DEV_PREDICTIONS.update(EXTERNAL_DEV_PREDICTIONS)
print("Development predictions to compare:", DEV_PREDICTIONS)


## Pre-training comparison and approval gate

Generate a reproducible development-set score table and charts (sacreBLEU, chrF++, review flags). Predictions from every model must match the **same IDs, source, reference and metadata**; otherwise scoring refuses to compare them. Small samples are only smoke tests, not scientific evidence. Review per-language/script quality and the diagnostics before training. For real data, explicitly set `APPROVE_REAL_TRAINING=True` **after** validation; this prevents accidental finetuning during exploration. The held-out test set is reserved until after training.


In [ ]:
from bhaashik.pretrain_review import compare_dev, plot_review
if DEV_PREDICTIONS:
    DEV_REVIEW = compare_dev(DEV_SAMPLE, DEV_PREDICTIONS, REVIEW_DIR)
    for model_result in DEV_REVIEW["models"]:
        print(model_result["model"], "chrF++", model_result["chrfpp"],
              "BLEU", model_result["bleu"], "flags", model_result["review_flags"])
    for chart in plot_review(REVIEW_DIR):
        print("Saved figure:", chart)
        try:
            from IPython.display import display, Image
            display(Image(filename=str(chart)))
        except ImportError:
            pass
else:
    print("No baseline predictions yet; data diagnostics are available above.")

APPROVE_REAL_TRAINING = False  # Set True only after reviewing real data
TRAIN_APPROVED = USE_EXAMPLE_DATA or (
    APPROVE_REAL_TRAINING
    and CORPUS_REVIEW["ready_for_training"]
    and bool(DEV_PREDICTIONS)
)
print("Training approved:", TRAIN_APPROVED)


## Finetune and export a normal HF checkpoint (requires approval for real data)

Set `APPROVE_REAL_TRAINING=True` in the earlier review cell only after checking diagnostics and native model coverage. The default LoRA configuration uses batch 1, gradient accumulation 16, and sequence length 128. For resumability preserve `RUN/checkpoints/checkpoint-N` separately; the fast final-results archive excludes Trainer snapshots unless requested. Use a fresh run directory per seed/model/data split.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.modeling import train
    config = dict(model=MODEL, revision=REVISION, src_lang=SRC, tgt_lang=TGT,
                  train_file=str(DATA/'train.jsonl'), dev_file=str(DATA/'dev.jsonl'), output_dir=str(RUN),
                  mode=MODE, seed=SEED, epochs=3, learning_rate=2e-4 if MODE=='lora' else 2e-5,
                  batch_size=1, gradient_accumulation_steps=16, max_length=128,
                  precision='fp16', gradient_checkpointing=True, lora_rank=8)
    config_path = Path('configs') / f'{PAIR}-{MODE}-s{SEED}.json'
    config_path.write_text(json.dumps(config, indent=2))
    print(train(config_path))
    (RUN / "training_config.json").write_text(json.dumps(config, indent=2))
else:
    print("Skipped training. Complete dev review and explicitly approve real-data training.")


In [ ]:
if TRAIN_APPROVED:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    # LoRA is merged; no PEFT wrapper/custom architecture is needed for consumers.
    tokenizer = AutoTokenizer.from_pretrained(RUN)
    model = AutoModelForSeq2SeqLM.from_pretrained(RUN)
    print(type(model).__name__, len(tokenizer))
    del model, tokenizer
    import gc
    gc.collect(); torch.cuda.empty_cache()
    translate(RUN, SRC, TGT, DATA/'test.jsonl', RUN/'predictions.jsonl',
              batch_size=1, max_length=128, max_new_tokens=128, beams=4)
else:
    print("Skipped post-training translation; real-data training not approved.")


## Held-out test evaluation after training

This is the first point where the frozen **test** split is translated. It runs the untouched NLLB baseline and then compares the trained system with the same IDs/references. Do not inspect test scores while selecting training configurations or choosing hyperparameters.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.modeling import translate
    BASE = Path('runs') / f'{PAIR}-baseline'
    baseline_path = BASE / 'predictions.jsonl'
    if not baseline_path.exists():
        translate(MODEL, SRC, TGT, DATA/'test.jsonl', baseline_path, revision=REVISION,
                  batch_size=1, max_length=128, max_new_tokens=128, beams=4)
    print("Held-out test baseline:", baseline_path)
else:
    BASE = Path('runs') / f'{PAIR}-baseline'
    print("Test baseline not run: real-data training not approved.")


## Paired evaluation and human review

Corpus BLEU/chrF++ are not dialect-fidelity judgments. Bootstrap whole documents when IDs exist. A fixture-sized sample cannot support a paper claim; add qualified native-speaker annotation and multiple training seeds. Exported flags are triage hints, not automatic linguistic diagnoses. Import `evaluation.json` in the Pages portal to view actual scores.

In [ ]:
if TRAIN_APPROVED:
    from bhaashik.evaluation import evaluate, export_errors
    result = evaluate(RUN/'predictions.jsonl', RUN/'evaluation.json',
                      baseline=BASE/'predictions.jsonl', samples=1000, seed=SEED)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    export_errors(RUN/'predictions.jsonl', RUN/'annotations.csv', seed=SEED)
else:
    print("Final test evaluation skipped (no trained model).")


## Preserve artifacts

The fast archive stores **only the current experiment**, the corresponding baseline and the training configuration. It preserves the exported HF model/tokenizer, predictions, manifests, metrics and annotations. By default it **excludes the large Trainer checkpoint directory**, so it is suitable for preserving final results but **cannot be used to resume training**. Set `INCLUDE_CHECKPOINTS=True` if you require optimizer/scheduler checkpoint state for resuming. ZIP files use no additional compression and can therefore be larger, but are quicker to create; model weights may already be internally efficient. All outputs remain under the same established repository structure. Dataset contents are not copied into this archive; preserve your licensed data separately. Save the Kaggle version with outputs or copy/download the ZIP before ending the session.


In [ ]:
if TRAIN_APPROVED:
    from bhaashik.artifacts import archive_experiment
    
    # False: fast final-results archive; True: include optimizer/trainer checkpoints for resuming
    INCLUDE_CHECKPOINTS = False
    archive_path = ROOT.parent / f'bhaashik-{PAIR}-{MODE}-s{SEED}-artifacts.zip'
    archive = archive_experiment(
        ROOT, RUN, archive_path, baseline_dir=BASE, config_path=config_path,
        include_checkpoints=INCLUDE_CHECKPOINTS, report_dir=REVIEW_DIR,
    )
    print('Download or copy to persistent storage:', archive)
    print('Size (GiB):', round(archive.stat().st_size / 2**30, 2))
    # Colab: from google.colab import files; files.download(str(archive))
    # Kaggle: use the notebook Output files panel.
else:
    print("No trained run to archive. Pretraining review reports are in:", REVIEW_DIR)
